In [1]:
import json
import shutil
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from torch.fx.experimental.unification.multipledispatch.dispatcher import source

load_dotenv()

C:\2026-Projects\Document_Intelligent_Hub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT/"data"/"processed"
VECTOR_STORE_DIR =PROJECT_ROOT / "data" / "vectore_store" / "chroma"

CHUNKED_DOCUMENTS_PATH = PROCESSED_DATA_DIR / "chunked_documents.json"

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Chunked documents path: {CHUNKED_DOCUMENTS_PATH}")
print(f"Vector store path: {VECTOR_STORE_DIR}")

Chunked documents path: C:\2026-Projects\Document_Intelligent_Hub\notebooks\data\processed\chunked_documents.json
Vector store path: C:\2026-Projects\Document_Intelligent_Hub\notebooks\data\vectore_store\chroma


In [3]:
def json_records_to_documents(records: list[dict]) -> list[Document]:
    documents = []

    for record in records:
        documents.append(
            Document(
                page_content=record["page_content"],
                metadata=record["metadata"],
            )
        )
    return documents

In [4]:
if not CHUNKED_DOCUMENTS_PATH.exists():
    raise FileNotFoundError(
        f"Chunked documents file not found: {CHUNKED_DOCUMENTS_PATH}"
        "Run 02_document_preprocessing.ipynb first."
    )

with CHUNKED_DOCUMENTS_PATH.open("r") as file:
    chunked_records = json.load(file)

    chunked_documents = json_records_to_documents(chunked_records)
    print(f"Loaded {len(chunked_documents)} chunked documents.")

Loaded 5 chunked documents.


In [5]:
if chunked_documents:
    print(chunked_documents[0].metadata)
    print(chunked_documents[0].page_content[:500])

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'moddate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'department': 'Compliance', 'access_role': 'Compliance Analyst', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'page_number': 1, 'part_index': 0, 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'chunk_index': 0, 'chunk_size': 991}
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party

In [6]:
EMBEDDING_MODEL_NAME ="sentence-transformers/all-MiniLM-L6-v2"

embedding_function = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8869.82it/s]


In [7]:
#!!ONLY RUN THIS IF I WANT TO CLEAN THE EXISTING VECTOR DATABASE!!

RESET_VECTOR_STORE = False # swich to true when doing clean ups.

if RESET_VECTOR_STORE and VECTOR_STORE_DIR.exists():
    shutil.rmtree(VECTOR_STORE_DIR)
    VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
    print("Existing vector store deleted.")

Existing vector store deleted.


In [8]:
def get_document_ids(documents: list[Document]) -> list[str]:
    ids = []

    for index, document in enumerate(documents):
        chunked_id = document.metadata.get("chunked_id")

        if chunked_id:
            ids.append(chunked_id)
        else:
            source = document.metadata.get("source", "unknown_source")
            ids.append(f"{source}_{index}")

    return ids

In [9]:
document_ids = get_document_ids(chunked_documents)

print(f"Generated {len(document_ids)} document IDs.")
print(document_ids[:5])

Generated 5 document IDs.
['Sample Compliance Manual.pdf_0', 'Sample Compliance Manual.pdf_1', 'Sample Compliance Manual.pdf_2', 'Sample Compliance Manual.pdf_3', 'Sample Compliance Manual.pdf_4']


In [10]:
COLLECTION_NAME = "enterprise_documents"

vector_store = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embedding_function,
    ids=document_ids,
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
)

print("Vector store created.")
print(f"Persisted diractory: {VECTOR_STORE_DIR}")

Vector store created.
Persisted diractory: C:\2026-Projects\Document_Intelligent_Hub\notebooks\data\vectore_store\chroma


In [11]:
collection_count = vector_store._collection.count()
print(f"Documents in Chroma collection: {collection_count}")

Documents in Chroma collection: 5


In [12]:
reloaded_vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_function,
    persist_directory=str(VECTOR_STORE_DIR),
)

reloaded_count = reloaded_vector_store._collection.count()

print(f"Reloaded Chroma collection count: {reloaded_count}")

Reloaded Chroma collection count: 5


In [13]:
query = "What are the reporting requiremnets?"

results = reloaded_vector_store.similarity_search(
    query = query,
    k=5,
)

print(f"Query: {query}")
print(f"Results: {results}")

Query: What are the reporting requiremnets?
Results: [Document(id='Sample Compliance Manual.pdf_1', metadata={'chunk_size': 498, 'creationdate': '2026-06-20T21:35:22+02:00', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'access_role': 'Compliance Analyst', 'total_pages': 3, 'file_type': 'pdf', 'source': 'Sample Compliance Manual.pdf', 'chunk_id': '40b1525f-5997-4905-993a-07e5013123b1', 'moddate': '2026-06-20T21:35:22+02:00', 'creator': 'Microsoft® Word for Microsoft 365', 'part_index': 0, 'page': 0, 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'author': 'BENNET DYANI', 'page_label': '1', 'department': 'Compliance', 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'chunk_index': 1, 'producer': 'Microsoft® Word for Microsoft 365', 'page_number': 1}, page_content='• Transactions involving foreign accounts must be flagged for enhanced due \ndiligence. \nSuspicious Activity Indicators \nEmployees must be alert to:

In [14]:
for index, document in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {index}:")
    print("-"* 80)
    print("Source:", document.metadata.get("source"))
    print("Page", document.metadata.get("page_number"))
    print("Chunked ID:", document.metadata.get("chunked_id"))
    print()
    print(document.page_content[:1000])

Result 1:
--------------------------------------------------------------------------------
Source: Sample Compliance Manual.pdf
Page 1
Chunked ID: None

• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employees. 
• Certification records must be retained for 5 years.
Result 2:
--------------------------------------------------------------------------------
Source: Sample Compliance Manual.pdf
Page 1
Chunked ID: None

Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management ac

In [15]:
scored_results = reloaded_vector_store.similarity_search_with_relevance_scores(
    query = query,
    k=5,
)

for index, (document, score) in enumerate(scored_results, start=1):
    print("=" * 80)
    print(f"Result {index}:")
    print(f"Relevance score: {score:.4f}")
    print("Source:", document.metadata.get("source"))
    print("Page", document.metadata.get("page_number"))
    print("Chunked ID:", document.metadata.get("chunked_id"))
    print()
    print( document.page_content[:750])


Result 1:
Relevance score: 0.2814
Source: Sample Compliance Manual.pdf
Page 1
Chunked ID: None

• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employees. 
• Certification records must be retained for 5 years.
Result 2:
Relevance score: 0.2076
Source: Sample Compliance Manual.pdf
Page 1
Chunked ID: None

Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organizat

In [16]:
test_queries = [
    "What are the reporting requirements?",
     "AML threshold requirements",
    "suspicious activity reporting",
    "customer due diligence policy",
    "data retention requirements",
    "privacy breach notification",
    "third party risk management",

]

In [17]:
for query in test_queries:
    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)

    results = reloaded_vector_store.similarity_search_with_relevance_scores(
        query = query,
        k=3,
    )

    for index, (document, score) in enumerate(results, start=1):
        print(f"Result {index} | Score: {score:.4f}")
        print("Source:", document.metadata.get("source"))
        print("Page:", document.metadata.get("page_number"))
        print(document.page_content[:500])

QUERY: What are the reporting requirements?
Result 1 | Score: 0.3021
Source: Sample Compliance Manual.pdf
Page: 1
• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employees. 
• Certification records must be retained for 5 years.
Result 2 | Score: 0.2868
Source: Sample Compliance Manual.pdf
Page: 3
• Contracts must include clauses requiring adherence to organizational 
compliance policies. 
Incident Reporting 
• Operational incidents must be logged within 24 hours. 
• Escalation procedures must be followed for severe disruptions. 
 
Page 5 — Audit & Enforcement 
Internal Audits 
• Conducted semi-annually by th

C:\Users\benne\AppData\Local\Temp\ipykernel_96\1254288254.py:6: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_0', metadata={'file_type': 'pdf', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'creationdate': '2026-06-20T21:35:22+02:00', 'chunk_size': 991, 'creator': 'Microsoft® Word for Microsoft 365', 'source': 'Sample Compliance Manual.pdf', 'access_role': 'Compliance Analyst', 'chunk_index': 0, 'total_pages': 3, 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'author': 'BENNET DYANI', 'page': 0, 'moddate': '2026-06-20T21:35:22+02:00', 'producer': 'Microsoft® Word for Microsoft 365', 'department': 'Compliance', 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'part_index': 0, 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'page_label': '1', 'page_number': 1}, page_content='Sample Compliance Manual \n \nPage 1 — Introduction & Scope \nThis Compliance M

In [19]:
filtered_results = reloaded_vector_store.similarity_search(
    query = "reporting obligations",
    k=5,
    filter={
        "department": "Compliance",
    },
)

for document in filtered_results:
    print(document.metadata.get("source"))
    print(document.page_content[:500])
    print()


Sample Compliance Manual.pdf
• Contracts must include clauses requiring adherence to organizational 
compliance policies. 
Incident Reporting 
• Operational incidents must be logged within 24 hours. 
• Escalation procedures must be followed for severe disruptions. 
 
Page 5 — Audit & Enforcement 
Internal Audits 
• Conducted semi-annually by the Internal Audit Department. 
• Findings must be documented and shared with senior management. 
External Audits 
• Regulators may initiate audits at any time. 
• Full cooperation is r

Sample Compliance Manual.pdf
• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employe